# LSTM Experiment 2

## Overview

The first LSTM experiment established a baseline deep learning model for burger sales forecasting.

Although the model successfully learned sequential data, its predictive performance was considerably lower than the LightGBM model.

This experiment introduces several improvements specifically designed for neural networks:

- MinMax feature scaling
- Separate scaling of the target variable
- Improved preprocessing pipeline
- Improved training strategy

The objective is to evaluate whether these modifications improve forecasting accuracy.

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import (
    MinMaxScaler,
    LabelEncoder
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

print("TensorFlow Version:", tf.__version__)
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import (
    MinMaxScaler,
    LabelEncoder
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0
TensorFlow Version: 2.21.0


# Loading the Dataset

The cleaned dataset generated during the preprocessing stage is loaded.

All feature engineering completed in previous notebooks is preserved.

In [ ]:
# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("../data/burger_data_processed.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (24500, 8)


,Date,Region,Temperature,Humidity,Wind,Visibility,Pressure,Sales
0,15-09-2020,Reg1,10.248814,0.779164,11.509130,14.503403,1017.293917,991.60
1,14-09-2020,Reg1,10.337595,0.908549,7.432656,2.232960,1019.452636,1858.59
2,13-09-2020,Reg1,20.763686,0.505324,7.788249,4.779211,1022.677119,3.99
3,12-09-2020,Reg1,21.500892,0.758557,3.767432,9.904534,1009.341357,3090.78
4,11-09-2020,Reg1,21.774269,0.398296,20.705369,15.224605,1015.713234,990.99


# Convert Date Column

The Date column is converted into datetime format to preserve time-series information.

In [375]:
# ==========================================================
# Convert Date
# ==========================================================

df["Date"] = pd.to_datetime(df["Date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24500 entries, 0 to 24499
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         24500 non-null  datetime64[ns]
 1   Region       24500 non-null  object        
 2   Temperature  24500 non-null  float64       
 3   Humidity     24500 non-null  float64       
 4   Wind         24500 non-null  float64       
 5   Visibility   24500 non-null  float64       
 6   Pressure     24500 non-null  float64       
 7   Sales        24500 non-null  float64       
dtypes: datetime64[ns](1), float64(6), object(1)
memory usage: 1.5+ MB


# Encode Region

The Region column contains categorical values.

Neural networks require numerical inputs, so the region labels are converted into integer values.

In [376]:
# ==========================================================
# Encode Region
# ==========================================================

encoder = LabelEncoder()

df["Region"] = encoder.fit_transform(df["Region"])

print(encoder.classes_)

['Reg1' 'Reg10' 'Reg2' 'Reg3' 'Reg4' 'Reg5' 'Reg6' 'Reg7' 'Reg8' 'Reg9']


# Feature Selection

The target variable is Sales.

The Date column is excluded from the feature matrix because temporal information has already been represented through engineered calendar features.

In [377]:
# ==========================================================
# Feature Selection
# ==========================================================

X = df.drop(
    columns=[
        "Sales",
        "Date"
    ]
)

y = df["Sales"]

print("Features:", X.shape)

print("Target:", y.shape)

Features: (24500, 6)
Target: (24500,)


# Train, Validation and Test Split

The dataset is divided chronologically to preserve the temporal order of observations.

Random shuffling is avoided because future observations should never influence model training.

In [378]:
# ==========================================================
# Chronological Split
# ==========================================================

train_size = int(len(df) * 0.70)

validation_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]

X_validation = X.iloc[
    train_size:
    train_size + validation_size
]

X_test = X.iloc[
    train_size + validation_size:
]

y_train = y.iloc[:train_size]

y_validation = y.iloc[
    train_size:
    train_size + validation_size
]

y_test = y.iloc[
    train_size + validation_size:
]

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Testing:", X_test.shape)

Training: (17150, 6)
Validation: (3675, 6)
Testing: (3675, 6)
